# Healthcare Information Chatbot Using LLMs and RAG

This notebook walks through the project in small steps using the local `data/health.pdf` manual. It uses CPU-compatible embeddings, FAISS, TF-IDF, and `google/flan-t5-base`. The source is the WHO 2017 hypertension manual; this project is educational, not medical advice.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "health.pdf").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project folder: {PROJECT_ROOT}")

Project folder: C:\Users\SHRAVANI\OneDrive\Desktop\RAG


## 1. Load the PDF and check its pages

`PyPDFLoader` returns one document per PDF page. The printed page label is kept for answer citations.

In [2]:
from src.pdf_loader import load_healthcare_pdf

pages = load_healthcare_pdf(PROJECT_ROOT / "data" / "health.pdf")
print(f"Number of PDF pages: {len(pages)}")
print("First page label:", pages[0].metadata.get("page_label"))
print(pages[0].page_content[:300])

Number of PDF pages: 38
First page label: i
Diagnosis
and management
for patients with hypertension
A noncommunicable disease education manual for primary health care professionals and patients


## 2. Split the manual into overlapping chunks

The splitter uses 500 characters per chunk and 100 characters of overlap. Reference lists are removed, while page numbers and bullet text are preserved.

In [3]:
from src.text_chunking import split_documents

chunks = split_documents(pages, chunk_size=500, chunk_overlap=100)
print(f"Searchable chunks: {len(chunks)}")
print("Example chunk page:", chunks[0].metadata["page_numbers"])
print(chunks[0].page_content)

Searchable chunks: 39
Example chunk page: ['1']
Blood pressure
Systolic
blood pressure
Diastolic
blood pressure
When the heart
contracts to
export blood
When the heart
relaxes to
import blood
120/80
mmHg


## 3. Create sentence embeddings and a FAISS index

`all-MiniLM-L6-v2` turns each chunk into a vector. The model and FAISS search run on CPU.

In [4]:
from src.embeddings import load_embedding_model
from src.vector_store import create_faiss_index

embedding_model = load_embedding_model()
faiss_index = create_faiss_index(
    [chunk.page_content for chunk in chunks],
    embedding_model,
)
print(f"FAISS vectors: {faiss_index.ntotal}")

FAISS vectors: 39


## 4. Set up keyword and hybrid retrieval

TF-IDF finds exact words and phrases. The retriever combines TF-IDF ranks with semantic FAISS ranks and joins relevant adjacent chunks from the same page.

In [5]:
from src.retrieval import HybridRetriever

retriever = HybridRetriever(chunks, faiss_index, embedding_model)
print(f"TF-IDF documents: {retriever.keyword_matrix.shape[0]}")
print(f"TF-IDF terms: {retriever.keyword_matrix.shape[1]}")

TF-IDF documents: 39
TF-IDF terms: 1219


## 5. Try a question and inspect the retrieved evidence

Always look at the retrieved passages and page references before generating an answer.

In [6]:
question = "What are the symptoms of hypertension?"
retrieved = retriever.retrieve(question)
for result in retrieved:
    print("Source pages:", ", ".join(result.metadata["page_numbers"]))
    print(result.page_content)
    print("-" * 60)

Source pages: 5
Symptoms of hypertension
People with high blood pressure most of the time
DO NOT HAVE ANY SYMPTOMS
Possible symptoms include:
headache
palpitation
excess sweating
------------------------------------------------------------
Source pages: 6
Patient education Professional information
• Most people think of a stiff neck, headache, 
or dizziness as common symptoms of 
hypertension.
• However, most patients with hypertension have 
no symptoms unless their blood pressure is 
very high.
• As such, patients are tempted to skip their 
regular check-ups or even their medication 
because they lack symptoms.
• However, untreated hypertension with no 
symptoms can cause stroke, heart failure, 
myocardial infarction, or even death.
• This is why hypertension is called a “silent 
killer” because it can end your life suddenly 
without warning signs.
• It should be emphasized that most patients 
have no symptoms.
• Secondary hypertension caused by 
phaeochromocytoma or Cushing’s syndrom

## 6. Generate an answer with FLAN-T5-base

The prompt is limited to 512 input tokens. The model is loaded once and kept on CPU.

In [7]:
from src.chatbot import (
    DISCLAIMER,
    NOT_FOUND_MESSAGE,
    _format_source_pages,
    _prepare_context,
    generate_answer,
)

if retrieved:
    context, used_documents = _prepare_context(retrieved)
    answer = generate_answer(question, context)
    print(answer)
    print(_format_source_pages(used_documents))
else:
    print(NOT_FOUND_MESSAGE)
print(DISCLAIMER)

headache palpitation excess sweating Patient education Professional information • Most people think of a stiff neck, headache, or dizziness as common symptoms of hypertension. • However, most patients with hypertension have no symptoms unless their blood pressure is very high.
Source pages: 5, 6
Medical disclaimer: This is an educational student project based only on the provided WHO 2017 manual. It is not medical advice or a personal diagnosis, and the manual may not reflect current guidelines. Consult a qualified healthcare professional about personal medical concerns.


## 7. Ask another question

The same retriever and loaded models can be reused for each question.

In [8]:
next_question = "How is hypertension diagnosed according to the document?"
next_results = retriever.retrieve(next_question)
if next_results:
    next_context, next_sources = _prepare_context(next_results)
    print(generate_answer(next_question, next_context))
    print(_format_source_pages(next_sources))
else:
    print(NOT_FOUND_MESSAGE)

When the average systolic blood pressure is 140 mmHg or higher, or when the average diastolic blood pressure is 90 mmHg or higher, taken on two or more separate days.
Source pages: 4, 8
